# Permutation-based search (BOSS / GRaSP / BOSS-FCI / GRaSP-FCI / IGSP / UT-IGSP)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/permutation_based.ipynb)

BOSS and GRaSP search over orderings of the variables. For an ordering, a score, by default the
extended BIC (`ebic_gamma=1.0`), chooses each variable's parents from the variables before it; the
search keeps the ordering with the best score. Both return a CPDAG, like GES, and suit
linear-Gaussian data. BOSS-FCI and GRaSP-FCI start from their graph and allow hidden common causes,
returning a PAG, as GFCI does from GES's. IGSP and UT-IGSP search orderings too, on data from several
environments: IGSP when it is told the changed variables, UT-IGSP when it is told them in part or
not at all.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.14.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.14.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 2,000 samples.

In [3]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

2000 observations, 10 variables, seed 7


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 332" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="332" fill="#fafbfc"/><line x1="312.82" y1="44.82" x2="242.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,39.30 310.06,40.65 315.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="144.93" y1="163.00" x2="409.13" y2="134.59" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.07,133.52 408.60,129.62 409.67,139.56" fill="#3a4658" stroke-linejoin="round"/><line x1="113.24" y1="177.70" x2="42.60" y2="227.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,171.91 110.35,173.62 116.14,181.78" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="151.70" x2="42.84" y2="105.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,157.22 115.57,147.53 110.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="128.29" x2="244.82" y2="100.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,129.96 410.16,123.36 408.49,133.22" fill="#3a4658" stroke-linejoin="round"/><line x1="410.18" y1="124.86" x2="144.33" y2="36.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,128.02 411.76,120.11 408.60,129.60" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="139.19" x2="344.37" y2="160.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,136.12 408.58,134.43 411.65,143.95" fill="#3a4658" stroke-linejoin="round"/><line x1="139.66" y1="224.85" x2="218.45" y2="116.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="224.34,108.76 214.41,113.89 222.49,119.79" fill="#3a4658" stroke-linejoin="round"/><line x1="142.92" y1="242.40" x2="212.67" y2="287.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.08,292.82 215.38,283.20 209.96,291.60" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="235.35" x2="45.00" y2="235.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,235.35 109.00,230.35 109.00,240.35" fill="#3a4658" stroke-linejoin="round"/><line x1="212.82" y1="85.57" x2="142.84" y2="39.30" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.16,91.09 215.57,81.40 210.06,89.74" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="105.43" x2="312.82" y2="151.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,157.22 315.57,147.53 310.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="98.26" x2="45.00" y2="98.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,98.26 209.00,93.26 209.00,103.26" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="44.82" x2="42.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,39.30 110.06,40.65 115.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="292.81" x2="44.37" y2="239.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,295.88 211.65,288.05 208.58,297.57" fill="#3a4658" stroke-linejoin="round"/><circle cx="332.00" cy="32.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="32.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="132.00" cy="164.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="164.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="132.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="132.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominan

## Fit BOSS and GRaSP

BOSS starts from an ordering computed from the data and visits the variables in a random order;
GRaSP starts from a random ordering. `seed` fixes the random choices, so a run repeats exactly;
without `seed`, the `ANDREY_SEED` environment variable sets it (default 0). `metadata["score"]` is
the extended BIC of the final ordering, which the search minimizes. A larger `penalty_discount`
weighs the BIC penalty more and gives a sparser graph.

In [4]:
results = {
    "BOSS": andrey.boss(X, seed=0),
    "GRaSP": andrey.grasp(X, seed=0),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}")
    display(andrey.viz.draw(result.structure))

BOSS: cpdag, score -22359.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

GRaSP: cpdag, score -22359.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

## Compare with GES over ten datasets

GES searches over CPDAGs with the extended BIC as well. Structural Hamming distance (SHD) counts
errors against the truth's CPDAG; lower is better. The empty graph's SHD is the number of true
edges, so an estimate at or above it has not improved on returning no edges. One dataset can
favor any method, so the table covers ten seeds and their median.

In [5]:
methods = {
    "GES": lambda X: andrey.ges(X),
    "BOSS": lambda X: andrey.boss(X, seed=0),
    "GRaSP": lambda X: andrey.grasp(X, seed=0),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="cpdag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

seed,0,1,2,3,4,5,6,7,8,9,median
Empty graph,15,15,15,15,15,15,15,15,15,15,15.0
GES,21,10,0,10,17,7,11,17,0,0,10.0
BOSS,6,8,0,10,1,9,0,6,0,0,3.5
GRaSP,0,0,0,9,1,0,0,6,0,0,0.0


## Hidden common causes: BOSS-FCI and GRaSP-FCI

BOSS-FCI and GRaSP-FCI run BOSS's or GRaSP's search, remove the adjacencies that independence
tests explain by a hidden common cause, and orient the rest with FCI's rules, so they return a PAG
(partial ancestral graph), as GFCI does from the GES graph. Here `c` and `d` share a hidden cause,
and `a` and `b` each cause one of them; the PAG marks `c <-> d`.

In [6]:
HIDDEN = [("h", "c"), ("h", "d"), ("a", "c"), ("b", "d")]  # h has no column


def sample_hidden(seed: int, n: int = 2000, shuffle: bool = True) -> pd.DataFrame:
    """Draw n rows of a, b, c, d, whose hidden common cause h is left out."""
    rng = np.random.default_rng(seed)
    hidden = rng.standard_normal(n)
    a = rng.standard_normal(n)
    b = rng.standard_normal(n)
    c = 0.8 * a + 0.8 * hidden + rng.standard_normal(n)
    d = 0.8 * b + 0.8 * hidden + rng.standard_normal(n)
    frame = pd.DataFrame({"a": a, "b": b, "c": c, "d": d})
    return frame[rng.permutation(frame.columns)] if shuffle else frame


frame = sample_hidden(seed=3)
hybrids = {"BOSS-FCI": andrey.boss_fci(frame, seed=0), "GRaSP-FCI": andrey.grasp_fci(frame, seed=0)}
for name, result in hybrids.items():
    print(f"{name}: {result.structure.oriented_edges()}")
andrey.viz.draw(hybrids["BOSS-FCI"].structure)

BOSS-FCI: [('d', 'c', 'bidirected'), ('b', 'd', 'partially_directed'), ('a', 'c', 'partially_directed')]
GRaSP-FCI: [('d', 'c', 'bidirected'), ('b', 'd', 'partially_directed'), ('a', 'c', 'partially_directed')]


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="148.76" y1="142.17" x2="211.24" y2="97.83" stroke="#d6495b" stroke-width="2.0" stroke-linecap="round"/><polygon points="140.60,147.96 151.65,146.25 145.86,138.10" fill="#d6495b" stroke-linejoin="round"/><polygon points="219.40,92.04 208.35,93.75 214.14,101.90" fill="#d6495b" stroke-linejoin="round"/><line x1="153.00" y1="155.48" x2="208.60" y2="155.48" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><polygon points="143.00,155.48 153.00,160.48 153.00,150.48" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="212.80" cy="155.48" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="151.40" y1="84.52" x2="207.00" y2="84.52" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="147.20" cy="84.52" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><polygon points="217.00,84.52 207.00,79.52 207.00,89.52" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="130.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">d</text><circle cx="130.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">a</text><circle cx="230.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">c</text><circle cx="230.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">b</text></svg>'

### Compare with GFCI over ten datasets

`latent_confounded_er` hides one variable in ten of an Erdos-Renyi graph with linear-Gaussian
mechanisms. The truth is the PAG of the measured variables, from
`andrey.data.latent.marginal`, and the endpoint SHD counts each pair whose two endpoint marks
differ from it; lower is better, and the empty graph's SHD is the floor to beat.

In [7]:
from andrey.data.latent import marginal

methods = {
    "GFCI": lambda X: andrey.gfci(X),
    "BOSS-FCI": lambda X: andrey.boss_fci(X, seed=0),
    "GRaSP-FCI": lambda X: andrey.grasp_fci(X, seed=0),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("latent_confounded_er", d=22).sample(n=2000, seed=s, scale="standardize")
    pag = marginal(draw.graph, "pag")
    d = draw.data.shape[1]
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="pag")
    rows[s] = {"Empty graph": metrics.shd(empty, pag, endpoint_aware=True)}
    for name, fit in methods.items():
        rows[s][name] = metrics.shd(fit(draw.data).structure, pag, endpoint_aware=True)
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

seed,0,1,2,3,4,5,6,7,8,9,median
Empty graph,76,84,74,64,82,70,66,80,70,88,75.0
GFCI,53,54,45,44,44,34,26,40,30,55,44.0
BOSS-FCI,49,47,40,33,42,42,26,42,29,35,41.0
GRaSP-FCI,49,47,40,29,55,38,28,42,29,35,39.0


## Known targets: IGSP

`igsp` learns from an observational environment and settings that intervene on some variables,
its targets, when it is told all of them: `env` gives each row's environment, and `targets` maps
each environment to its targets, `[]` for the observational one. Here the last 1,000 rows cut `x1`
from `x0` and draw it afresh, and the fit is told that `x1` changed. It returns the interventional
essential graph, which directs every edge with exactly one end among a setting's targets. `x1`
is the only target here, so both of its edges are directed, where the observational rows alone
would leave the chain undirected. The drawing marks `x1`, the target, as a square, with a bar
across the edge the intervention cuts.

In [8]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges, cause first


def sample_cut(seed: int, n: int = 3000, shuffle: bool = True) -> tuple[pd.DataFrame, np.ndarray]:
    """Draw n rows of the chain, the last 1,000 cutting x1 from x0, and the environments."""
    rng = np.random.default_rng(seed)
    env = np.repeat(["observed", "cut x1"], [n - 1000, 1000])
    x0 = rng.standard_normal(n)
    x1 = np.where(env == "cut x1", 0.0, 0.8 * x0) + rng.standard_normal(n)
    x2 = 0.8 * x1 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2})
    frame = frame[rng.permutation(frame.columns)] if shuffle else frame
    return frame, env


frame, env = sample_cut(seed=1)
igsp_out = andrey.igsp(frame, env, targets={"observed": [], "cut x1": ["x1"]})
print(igsp_out.structure.oriented_edges())
display(andrey.viz.draw(igsp_out.structure, targets=["x1"]))

[('x0', 'x1', 'directed'), ('x1', 'x2', 'directed')]


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="158.50" y1="120.00" x2="93.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="168.50,120.00 158.50,115.00 158.50,125.00" fill="#3a4658" stroke-linejoin="round"/><line class="intervention" x1="154.50" y1="113.00" x2="154.50" y2="127.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><line x1="191.50" y1="120.00" x2="257.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,120.00 257.00,115.00 257.00,125.00" fill="#3a4658" stroke-linejoin="round"/><rect class="intervened" x="168.50" y="108.50" width="23.00" height="23.00" rx="2" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="80.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text><circle cx="280.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text></svg>'

## Unknown targets: UT-IGSP

`ut_igsp` learns from an observational environment and settings that intervene on some variables,
its targets, when it is told some of them or none: `env` gives each row's environment, and
`targets` maps each environment to its known targets, `[]` for the observational one and `None`
for a setting whose targets are unknown. Here the last 1,000 rows shift the mean of `x1`, and the
fit is not told which variable changed. It returns the interventional essential graph with each
setting's targets, known and learned, in `metadata["targets"]`. It learns `x1` as the only
target, so, as with IGSP, both of `x1`'s edges have exactly one end among the targets and are
directed. The drawing marks the targets it learned as squares.

In [9]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges, cause first


def sample_shift(seed: int, n: int = 3000, shuffle: bool = True) -> tuple[pd.DataFrame, np.ndarray]:
    """Draw n rows of the chain, the last 1,000 shifting the mean of x1, and the environments."""
    rng = np.random.default_rng(seed)
    env = np.repeat(["observed", "shift x1"], [n - 1000, 1000])
    x0 = rng.standard_normal(n)
    x1 = 0.8 * x0 + rng.standard_normal(n) + 2.0 * (env == "shift x1")
    x2 = 0.8 * x1 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2})
    frame = frame[rng.permutation(frame.columns)] if shuffle else frame
    return frame, env


frame, env = sample_shift(seed=1)
ut_out = andrey.ut_igsp(frame, env, targets={"observed": [], "shift x1": None})
print(ut_out.structure.oriented_edges())
print("targets:", ut_out.metadata["targets"])
display(andrey.viz.draw(ut_out.structure, targets=ut_out.metadata["targets"]["shift x1"]))

[('x0', 'x1', 'directed'), ('x1', 'x2', 'directed')]
targets: {'observed': [], 'shift x1': ['x1']}


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="158.50" y1="120.00" x2="93.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="168.50,120.00 158.50,115.00 158.50,125.00" fill="#3a4658" stroke-linejoin="round"/><line class="intervention" x1="154.50" y1="113.00" x2="154.50" y2="127.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><line x1="191.50" y1="120.00" x2="257.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,120.00 257.00,115.00 257.00,125.00" fill="#3a4658" stroke-linejoin="round"/><rect class="intervened" x="168.50" y="108.50" width="23.00" height="23.00" rx="2" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="80.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text><circle cx="280.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text></svg>'

## Use another dataset

Change `seed`, or the `d` and `n` passed to `benchmarks.scm` and `sample`, and rerun. The
[release cards](https://andrey.abel.ai/docs/benchmarks.html) compare BOSS and GRaSP with
causal-learn, BOSS-FCI and GRaSP-FCI with Tetrad, and IGSP and UT-IGSP with
graphical-model-learning.